# OLMo-2-1B × opc-sft-stage2 leaderboard — long-horizon (r=64, 9000 steps, ~225M tokens)

Long-horizon characterization from `docs/notes/polar_product/tight_chord_paper_plan.md` (Phase L). Cell: OLMo-2-1B × `opc-sft-stage2` (4 sub-configs concat, 436k docs → 150k packed slots @ seq=2048) × r=64 × global_batch=16 (batch=4 × accum=4) × packed_v1 × constant LR × α=r × all-linear × bf16 × compile × single-GPU Blackwell. `max_steps=9000` ≈ 225M unique content tokens (~295M slot-tokens at 75% fill).

Source label: **OLMo OPC eval JSONL logs**: `allenai/OLMo-2-0425-1B`; `data/opc_sft_stage2_all_packed_seq2048`; `packed_v1` / `packed_v1.1`; seq2048; 9k-step eval logs.

Diagnostic q_agree source: **OLMo OPC JSONL q_agree**: `optim_step.awc_q_agree_*` from `logs/chord_tight_slack_phase_L_1b_r256_lr2_blackwell` (r256, ns=5, lr={3e-3,1e-2}). These are JSONL diagnostics, not snapshot-registry rows.

Phase L LR sweep, seed=0, three optimizer arms. AdamW + chord-tight each pool three sub-sweeps (original L1, packed-v1.1 repack of same η values, η-extension); chord-tight-clean κ_sr=0.75 is a single sub-sweep added 2026-05-26 to test whether the packed_v1 Magicoder winner transfers.

- **AdamW**: η ∈ {3e-5, 1e-4, 3e-4, 1e-3, 3e-3}
- **chord-tight (polar, k=1)** (`adam-polar-product-lora-coupled-spectral-chord-tight`): η ∈ {3e-3, 1e-2, 3e-2, 1e-1, 3e-1}
- **chord-tight-clean κ_sr=0.75** (`adam-polar-product-lora-coupled-spectral-chord-tight-clean`, polar_method=ssc, ssc_kappa_solver=stable_rank, picard=2): η ∈ {1e-3, 3e-3, 1e-2, 3e-2, 1e-1}

Source log groups: `{adamw,chord_tight}_phase_L_lrsweep_r64_{blackwell,repack_blackwell,extension_blackwell}`, `chord_tight_clean_ssc_kappa075_phase_L_lrsweep_r64_blackwell`.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lora_playground.loader import load_runs
from lora_playground.plotting import compare_variants_figure

# Plain chord-tight k=2 (ns + polar_express) removed from the leaderboard:
# the 2/(ρ·s) cross-coupling coefficient (optim.py:5573) does not benefit
# from picard>1 (packed_v1 r=64 matched-isolation: k=1 0.5120 vs k=3
# 0.5122). Replaced by chord-tight-clean k=2 (1/η coupling, optim.py:5057)
# which on packed_v1 r=64 ns improves k=1 0.5110 → k=3 0.5072.
GROUPS = [
    'adamw_phase_L_lrsweep_r64_blackwell',
    'chord_tight_phase_L_lrsweep_r64_blackwell',
    'adamw_phase_L_lrsweep_r64_repack_blackwell',
    'chord_tight_phase_L_lrsweep_r64_repack_blackwell',
    'adamw_phase_L_lrsweep_r64_extension_blackwell',
    'chord_tight_phase_L_lrsweep_r64_extension_blackwell',
    'chord_tight_clean_ssc_kappa075_phase_L_lrsweep_r64_blackwell',  # κ_sr=0.75 robustness check (k=2)
    'chord_tight_clean_ssc_fixedc_c0p2_phase_L_lrsweep_r64_gpuxl_h200',  # fixed-c=0.2 comparator k=2 (on H200)
    'chord_tight_polar_express_phase_L_lrsweep_r64_gpuxl_h200',  # chord-tight k=1 polar_express @ 10 iters (legacy H200 — walltimed at step 750)
    'chord_tight_polar_express_phase_L_lrsweep_r64_gpuxl_h200_resub',  # chord-tight k=1 polar_express @ 10 iters (H200 resub — completed to step 9000)
    'chord_tight_polar_express_phase_L_lrsweep_r64_lr1e-3_blackwell',  # low-η tail extension (lr=1e-3) for k=1 polar_express — Blackwell (gpuxl 4-GPU-min blocked H200)
    'chord_tight_clean_ssc_kappa075_phase_L_lrsweep_r64_extension_blackwell',  # κ_sr=0.75 LR extension to {3e-1, 1e0}
    'chord_tight_clean_ns_phase_L_lrsweep_r64_k2_blackwell',                # chord-tight-clean k=2 ns (1/η coupling, picard=2)
    'chord_tight_clean_polar_express_phase_L_lrsweep_r64_k2_gpuxl_h200',    # chord-tight-clean k=2 polar_express (1/η coupling, picard=2) — on H200 (matches predecessor 6450610)
    'chord_tight_clean_ssc_kappa075_k1_phase_L_lrsweep_r64_blackwell',      # κ_sr=0.75 k=1 ablation (at k=1 cross-coupling = 0 → polar+SSC)
    'chord_tight_clean_ssc_fixedc_c0p2_k1_phase_L_lrsweep_r64_blackwell',   # c=0.2 k=1 ablation (at k=1 cross-coupling = 0 → polar+SSC)
]

runs = load_runs(where={'log_group': GROUPS},
                 logs_root='../logs', warn_cross_commit=False)

# Dedup: keep the longest-trajectory run per (optimizer, lr, ssc_kappa).
# Phase-L extension runs were resumed from checkpoints, producing two cfg
# events per cell that differ in `resume_from`/`checkpoint_dir`. The loader
# treats them as distinct series; for the leaderboard we want the single
# continuation that reaches the highest step. Adding ssc_kappa to the key
# keeps room for future κ values without collapsing them onto each other.
_dedup = {}
for cfg, evs in runs:
    key = (cfg['optimizer'], float(cfg['lr']), cfg.get('ssc_kappa'), cfg.get('ssc_c'), cfg.get('polar_method'), cfg.get('_derived', {}).get('effective_picard_iters', cfg.get('picard_iters_override')))
    last_step = evs[-1]['step'] if evs else -1
    prev = _dedup.get(key)
    if prev is None or last_step > prev[1]:
        _dedup[key] = ((cfg, evs), last_step)
runs = [v[0] for v in _dedup.values()]

print(f'loaded: {len(runs)} runs (after dedup)')
for cfg, evs in sorted(runs, key=lambda r: (r[0]['optimizer'], r[0].get('ssc_kappa') or 0, float(r[0]['lr']))):
    last = evs[-1]
    kappa_tag = f"κ={cfg['ssc_kappa']}" if cfg.get('ssc_kappa') is not None else ''
    eff_k = cfg.get('_derived', {}).get('effective_picard_iters', cfg.get('picard_iters_override'))
    k_tag = f"k={eff_k}" if eff_k is not None else ''
    print(f"  {cfg['optimizer']:55s} {kappa_tag:9s} {k_tag:4s} η={float(cfg['lr']):.0e}  "
          f"step={last['step']}  eval={last['eval_loss']:.4f}  "
          f"[{cfg.get('log_group', '?')}]")

## r=64 — k=1 family


In [ ]:
from lora_playground.plotting import canonical_label
from IPython.display import display

def picard_of(cfg):
    return cfg.get('_derived', {}).get('effective_picard_iters', cfg.get('picard_iters_override')) or 1

def render_panel(prefetched, picard, suptitle):
    """Canonical-labeled panel for one picard family (k=1 or k=2); AdamW always
    overlaid. canonical_label tags family/ns/picard/damping explicitly (so ns5/ns8,
    abs/ε_rel/κ_sr/c, k=1/k=2 never merge) and puts AdamW black + first;
    compare_variants_figure's assert_label_discriminates guard hard-errors on any
    residual silent merge. The k=1/k=2 split is now a picard filter, not a label list."""
    def keep(cfg):
        lbl = canonical_label(cfg)
        if lbl is None:
            return False
        return lbl == 'AdamW' or picard_of(cfg) == picard
    labeled = [(c, h) for c, h in prefetched if keep(c)]
    labels = {canonical_label(c) for c, _ in labeled}
    fig, tdf, sdf = compare_variants_figure(
        variants={l: {} for l in labels}, common_where={}, ref_label='AdamW',
        target_label='AdamW', sigma_ref=0.0017, suptitle=suptitle, figsize=(13, 5),
        max_steps=9000, allow_partial=True,
        prefetched_runs=labeled, variant_key=canonical_label)
    display(tdf.style.format('{:.4f}', na_rep='—'))
    display(sdf.style.format({'final': '{:.4f}', 'delta': '{:+.4f}',
                              'delta_sigma': '{:+.2f}σ', 'best_lr': '{:.0e}'}, na_rep='—'))
    plt.show()
    return tdf, sdf

render_panel(runs, 1, 'OLMo-2-1B × opc-sft-stage2 × r=64 × 9000 steps — k=1 family (Phase L, packed_v1[.1])')

## r=64 — k=2 (picard) family


In [ ]:
render_panel(runs, 2, 'OLMo-2-1B × opc-sft-stage2 × r=64 × 9000 steps — k=2 (picard) family')

## r=256 — k=1 family (rank-extension robustness)


In [ ]:
GROUPS_R256 = [
    'adamw_phase_L_lrsweep_r256_blackwell',
    'chord_tight_phase_L_lrsweep_r256_blackwell',
    'chord_tight_clean_ssc_kappa075_phase_L_lrsweep_r256_blackwell',
    'chord_tight_clean_ssc_fixedc_c0p2_phase_L_lrsweep_r256_gpuxl_h200',  # fixed-c=0.2 comparator k=2 (on H200)
    'chord_tight_polar_express_phase_L_lrsweep_r256_gpuxl_h200',  # chord-tight k=1 polar_express @ 10 iters (legacy H200 slow run)
    'chord_tight_polar_express_phase_L_lrsweep_r256_blackwell',  # chord-tight k=1 polar_express batched-Gram (Blackwell)
    'chord_tight_polar_express_phase_L_lrsweep_r256_lr1e-3_blackwell',  # low-η tail extension (lr=1e-3) for k=1 polar_express
    'chord_tight_clean_ssc_kappa075_phase_L_lrsweep_r256_extension_blackwell',  # κ_sr=0.75 LR extension to {3e-1, 1e0}
    'chord_tight_clean_ns_phase_L_lrsweep_r256_k2_blackwell',             # chord-tight-clean k=2 ns (1/η coupling, picard=2)
    'chord_tight_clean_polar_express_phase_L_lrsweep_r256_k2_blackwell',  # chord-tight-clean k=2 polar_express (1/η coupling, picard=2)
    'chord_tight_clean_ssc_kappa075_k1_phase_L_lrsweep_r256_blackwell',   # κ_sr=0.75 k=1 ablation (at k=1 cross-coupling = 0 → polar+SSC)
    'chord_tight_clean_ssc_fixedc_c0p2_k1_phase_L_lrsweep_r256_blackwell', # c=0.2 k=1 ablation (at k=1 cross-coupling = 0 → polar+SSC)
    'chord_tight_phase_L_r256_epsrel_ns_probe_blackwell_v2',  # \u03b5_rel relative-damping NS probe (ns\u2208{5,8}, eps=1e-2)
]

runs_r256 = load_runs(where={'log_group': GROUPS_R256},
                      logs_root='../logs', warn_cross_commit=False)

# Same dedup pattern as r=64: keep longest-trajectory per (optimizer, lr, ssc_kappa).
_dedup = {}
for cfg, evs in runs_r256:
    key = (cfg['optimizer'], float(cfg['lr']), cfg.get('ssc_kappa'), cfg.get('ssc_c'), cfg.get('polar_method'), cfg.get('_derived', {}).get('effective_picard_iters', cfg.get('picard_iters_override')))
    last_step = evs[-1]['step'] if evs else -1
    prev = _dedup.get(key)
    if prev is None or last_step > prev[1]:
        _dedup[key] = ((cfg, evs), last_step)
runs_r256 = [v[0] for v in _dedup.values()]

print(f'loaded: {len(runs_r256)} runs (after dedup)')
for cfg, evs in sorted(runs_r256, key=lambda r: (r[0]['optimizer'], r[0].get('ssc_kappa') or 0, float(r[0]['lr']))):
    last = evs[-1]
    kappa_tag = f"κ={cfg['ssc_kappa']}" if cfg.get('ssc_kappa') is not None else ''
    eff_k = cfg.get('_derived', {}).get('effective_picard_iters', cfg.get('picard_iters_override'))
    k_tag = f"k={eff_k}" if eff_k is not None else ''
    print(f"  {cfg['optimizer']:55s} {kappa_tag:9s} {k_tag:4s} η={float(cfg['lr']):.0e}  "
          f"step={last['step']}  eval={last['eval_loss']:.4f}  "
          f"[{cfg.get('log_group', '?')}]")

In [ ]:
render_panel(runs_r256, 1, 'OLMo-2-1B × opc-sft-stage2 × r=256 × 9000 steps — k=1 family (Phase L, packed_v1.1)')

## r=256 — k=2 (picard) family


In [ ]:
render_panel(runs_r256, 2, 'OLMo-2-1B × opc-sft-stage2 × r=256 × 9000 steps — k=2 (picard) family')

## r=256 — ε_rel damping-regime probe


In [ ]:
# ── ε_rel damping-regime probe (OLMo OPC r=256) ───────────────────────────
# Does opnorm-RELATIVE preconditioner damping (precond_delta_relative=True,
# eps=1e-2) beat the ABSOLUTE-damping baseline, and does it make higher-NS (ns=8)
# polar safer? polar-express (abs, ns=10) overlaid as a third contrast. All arms
# chord-tight k=1. canonical_label tags ns + damping explicitly, so ε_rel ns=5 vs
# ns=8 vs abs ns=5 vs polar-express never merge (the bug that hit this cell).
EPSREL_GROUPS = [
    'adamw_phase_L_lrsweep_r256_blackwell',                   # reference / speed target
    'chord_tight_phase_L_lrsweep_r256_blackwell',             # abs-damp (eps=1e-6) ns=5
    'chord_tight_phase_L_r256_epsrel_ns_probe_blackwell_v2',  # ε_rel (eps=1e-2) ns∈{5,8}
    'chord_tight_polar_express_phase_L_lrsweep_r256_blackwell',  # abs-damp polar-express ns=10
]
epsrel_runs = load_runs(where={'log_group': EPSREL_GROUPS},
                        logs_root='../logs', warn_cross_commit=False, quiet=True)
_d = {}
for cfg, evs in epsrel_runs:
    key = (cfg['optimizer'], float(cfg['lr']), cfg.get('muon_ns_steps'), cfg.get('polar_method'),
           cfg.get('precond_delta'), cfg.get('precond_delta_relative'),
           cfg.get('_derived', {}).get('effective_picard_iters', cfg.get('picard_iters_override')))
    ls = evs[-1]['step'] if evs else -1
    if key not in _d or ls > _d[key][1]:
        _d[key] = ((cfg, evs), ls)
epsrel_runs = [v[0] for v in _d.values()]
labeled = [(c, h) for c, h in epsrel_runs if canonical_label(c) is not None]
labels = {canonical_label(c) for c, _ in labeled}
fig, table_df_epsrel, summary_df_epsrel = compare_variants_figure(
    variants={l: {} for l in labels}, common_where={}, ref_label='AdamW', target_label='AdamW',
    sigma_ref=0.0017, figsize=(13, 5), max_steps=9000, allow_partial=True,
    suptitle='OLMo-2-1B × opc-sft-stage2 × r=256 — damping regime + NS method (chord-tight k=1; abs / ε_rel / polar-express)',
    prefetched_runs=labeled, variant_key=canonical_label)
display(table_df_epsrel.style.format('{:.4f}', na_rep='—'))
display(summary_df_epsrel.style.format({'final': '{:.4f}', 'delta': '{:+.4f}',
                                        'delta_sigma': '{:+.2f}σ', 'best_lr': '{:.0e}'}, na_rep='—'))
plt.show()